In [3]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 200
import h5py
from robustpade import pade_approx
from hubbard_sc_exp import HubbardED
from series import calculate_coeffs

In [5]:
def get_4site_plaquette_coefficients(U, beta, mu, max_order=4, N_points=256, r=1.0):
    print("Initializing ED for 4-site plaquette...")
    ed = HubbardED(n_orbitals=4)
    H0_blocks, Hk_blocks = [], []
    
    # Periodic 4-site ring (Plaquette)
    neighbors_and_t = [(0, 1, 1.0), (0, 2, 1.0), (1, 3, 1.0), (2, 3, 1.0)]
    
    for i, basis in enumerate(ed.basis):
        H0_blocks.append(U * ed.HU[i] + mu * ed.Hmu[i])
        Hk_blocks.append(ed.Hamiltonian_Hopping(neighbors_and_t, basis))

    def free_energy_persite(t_array):
        omega_array = np.zeros_like(t_array, dtype=complex)
        for idx, t in enumerate(t_array):
            Z_t = 1.0 + 0.0j # Vacuum
            for i in range(len(ed.basis)):
                H_t = H0_blocks[i] + t * Hk_blocks[i]
                Z_t += np.sum(np.exp(-beta * np.linalg.eigvals(H_t)))
            
            # DIVIDE BY ed.n_orbitals (4) TO GET PER-SITE FREE ENERGY
            omega_array[idx] = -(1.0 / (beta * ed.n_orbitals)) * np.log(Z_t)
        return omega_array

    print(f"Integrating on complex circle with radius r={r} and N={N_points} points...")
    return calculate_coeffs(N=N_points, f=free_energy_persite, max_order=max_order+1, r=r)

if __name__ == "__main__":
    U_val = 8.0; beta_val = 0.75; mu_val = 3.0
    
    coeffs = get_4site_plaquette_coefficients(U=U_val, beta=beta_val, mu=mu_val, max_order=6)
    
    print("\n--- Exact 4-Site Plaquette Free Energy (Per Site) ---")
    for order, c in enumerate(coeffs):
        if order % 2 == 0:
            print(f"Order {order}: {c.real: .12e}")

Initializing ED for 4-site plaquette...
Integrating on complex circle with radius r=1.0 and N=256 points...

--- Exact 4-Site Plaquette Free Energy (Per Site) ---
Order 0: -4.007484514422e+00
Order 2: -1.527123079876e-01
Order 4: -8.346331802924e-03
Order 6:  1.457376989683e-03
